In [ ]:
pip install requests beautifulsoup4

import requests
from bs4 import BeautifulSoup
import re

main_url = 'https://www.example.com/properties/'
response = requests.get(main_url)
soup = BeautifulSoup(response.content, 'html.parser')

# Assuming links to individual property pages are within <a> tags inside a specific div
property_links = []
for a_tag in soup.find_all('a', href=True):
    link = a_tag['href']
    if re.search(r'/properties/', link):  # Adjust the pattern as needed
        property_links.append(link)

addresses = []
for link in property_links:
    property_url = f"https://www.example.com{link}"
    response = requests.get(property_url)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Assuming the address is within a specific tag (e.g., <div> with class "address")
    address_div = soup.find('div', class_='address')
    if address_div:
        address = address_div.get_text(strip=True)
        addresses.append(address)

for table in soup.find_all('table'):
    for row in table.find_all('tr'):
        cells = row.find_all('td')
        if cells:
            address = cells[1].get_text(strip=True)  # Adjust index as per table structure
            addresses.append(address)

for address in addresses:
    print(address)

import csv

with open('real_estate_holdings.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['Address'])
    for address in addresses:
        writer.writerow([address])

import requests
from bs4 import BeautifulSoup
import re
import csv

def scrape_real_estate_holdings(base_url):
    # Step 1: Fetch the main page
    response = requests.get(base_url)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Step 2: Extract property links
    property_links = []
    for a_tag in soup.find_all('a', href=True):
        link = a_tag['href']
        if re.search(r'/properties/', link):  # Adjust pattern to match URLs
            property_links.append(link)

    addresses = []
    for link in property_links:
        property_url = f"https://www.example.com{link}"
        response = requests.get(property_url)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Step 3: Extract address details
        address_div = soup.find('div', class_='address')
        if address_div:
            address = address_div.get_text(strip=True)
            addresses.append(address)

    # Step 4: Save to CSV
    with open('real_estate_holdings.csv', mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['Address'])
        for address in addresses:
            writer.writerow([address])

# Example usage
scrape_real_estate_holdings('https://www.example.com/properties/')


In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
from google.colab import files

def extract_property_details(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')

    property_details = {}

    # Extract property name
    property_name = soup.find('h2', class_='border-bottom-thick mt-20 mb-10')
    if property_name:
        property_details['Property Name'] = property_name.get_text(strip=True)

    # Extract details from the table
    table = soup.find('table', class_='table lined non-responsive')
    if table:
        rows = table.find_all('tr')
        for row in rows:
            header = row.find('th')
            value = row.find('td', class_='right')
            if header and value:
                key = header.get_text(strip=True)
                value = value.get_text(strip=True)
                property_details[key] = value

    # Extract description text
    description = soup.find('p', class_='text-small pu-10 pl-10')
    if description:
        property_details['Description'] = description.get_text(strip=True)

    return property_details

def save_to_csv(properties):
    # Define CSV file name
    file_path = 'real_estate_properties.csv'

    # Define CSV file headers
    fieldnames = ['Property Name', 'Location', 'Number of Units', 'Year Built', 'Joint-Venture Partner', 'Description']

    # Write to CSV file
    with open(file_path, mode='w', newline='', encoding='utf-8') as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames, quoting=csv.QUOTE_ALL)
        writer.writeheader()
        for property in properties:
            writer.writerow({
                'Property Name': property.get('Property Name', ''),
                'Location': property.get('Location', ''),
                'Number of Units': property.get('Number of Units', ''),
                'Year Built': property.get('Year Built', ''),
                'Joint-Venture Partner': property.get('Joint-Venture Partner', ''),
                'Description': property.get('Description', '')
            })
    print(f"Data saved to {file_path}")

    # Download the CSV file
    files.download(file_path)

# Example usage
property_url = 'https://www.redhill.com/portfolio.html'  # Replace with the actual URL
details = extract_property_details(property_url)

# Example list of properties to save to CSV
properties = [details]

# Save to CSV and trigger download
save_to_csv(properties)



Data saved to real_estate_properties.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
from google.colab import files

def extract_properties_from_page(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Find all property blocks
    property_blocks = soup.find_all('div', class_='column width-6')

    properties = []

    for block in property_blocks:
        property_details = {}

        # Extract property name
        property_name = block.find('h2', class_='border-bottom-thick mt-20 mb-10')
        if property_name:
            property_details['Property Name'] = property_name.get_text(strip=True)

        # Extract details from the table
        table = block.find('table', class_='table lined non-responsive')
        if table:
            rows = table.find_all('tr')
            for row in rows:
                header = row.find('th')
                value = row.find('td', class_='right')
                if header and value:
                    key = header.get_text(strip=True)
                    value = value.get_text(strip=True)
                    property_details[key] = value

        # Extract description text
        description = block.find('p', class_='text-small pu-10 pl-10')
        if description:
            property_details['Description'] = description.get_text(strip=True)

        # Add property details to the list of properties
        properties.append(property_details)

    return properties

def save_to_csv(properties):
    # Define CSV file name
    file_path = 'real_estate_properties.csv'

    # Define CSV file headers
    fieldnames = ['Property Name', 'Location', 'Number of Units', 'Year Built', 'Joint-Venture Partner', 'Description']

    # Write to CSV file
    with open(file_path, mode='w', newline='', encoding='utf-8') as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames, quoting=csv.QUOTE_ALL)
        writer.writeheader()
        for property in properties:
            writer.writerow({
                'Property Name': property.get('Property Name', ''),
                'Location': property.get('Location', ''),
                'Number of Units': property.get('Number of Units', ''),
                'Year Built': property.get('Year Built', ''),
                'Joint-Venture Partner': property.get('Joint-Venture Partner', ''),
                'Description': property.get('Description', '')
            })
    print(f"Data saved to {file_path}")

    # Download the CSV file
    files.download(file_path)

# Example usage
property_url = 'https://www.redhill.com/portfolio.html'  # Replace with the actual URL
properties = extract_properties_from_page(property_url)

# Save to CSV and trigger download
save_to_csv(properties)


Data saved to real_estate_properties.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
from bs4 import BeautifulSoup
import pandas as pd
from google.colab import files

# Example HTML content
html_content = '''
<div class="content-bar">
                                                    <p>Over the past two decades, we have transacted and disposed of multiple real estate asset transactions multi family, office, strip centers, industrial, hospitality, and single family homes. Moving forward, the company will evolve and engage in the development of real estate assets. We have established “funds” for these real estate assets, and these funds have performed well, delivering annual high double-digit returns.</p>
<p>Here is a partial list of the portfolios and projects we have facilitated and/or for which we have provided advisory services:</p>
<p>The Ventura Equity Group LLC – A SFR fund for flipping houses in LA County – 2011-2015<br>
The Wilshire Capital Group LLC – A SFR fund for flipping houses in LA County – 2010-2014<br>
Disposition 200 SFR portfolio, 2014<br>
Disposition 115 SFR portfolio, 2013<br>
<a id="01" tabindex="-1"></a></p>
<h2>Tarpon Point, Cape Coral, FL</h2>
<p></p><div id="metaslider-id-72" style="width: 100%;" class="ml-slider-3-70-2 metaslider metaslider-nivo metaslider-72 ml-slider ms-theme-default" role="region" aria-roledescription="Slideshow" aria-label="Tarpon Point" tabindex="1">
    <div id="metaslider_container_72">
        <div class="slider-wrapper theme-default"><div class="ribbon"></div><div id="metaslider_72" class="nivoSlider"><img fetchpriority="high" decoding="async" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" height="300" width="640" alt="" class="slider-72 slide-73" style="width: 614.713px; visibility: hidden; display: inline;"><img decoding="async" src="https://mulhollandequity.com/wp-content/uploads/2016/02/TarponPoint-resized-640x300.jpg" height="300" width="640" alt="" class="slider-72 slide-76" style="width: 614.713px; visibility: hidden; display: inline;"><img class="nivo-main-image" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" alt="" style="height: 288.138px; width: 614.713px;"><div class="nivo-caption"></div><div class="nivo-directionNav"><a class="nivo-prevNav" rel="nofollow" aria-label="Previous Slide">&lt;</a><a class="nivo-nextNav" rel="nofollow" aria-label="Next Slide">&gt;</a></div><div class="nivo-slice" name="0" style="left: 0px; width: 614.713px; height: 288.138px; opacity: 1; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-0px;"></div><div class="nivo-slice" name="1" style="left: 41px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-41px;"></div><div class="nivo-slice" name="2" style="left: 82px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-82px;"></div><div class="nivo-slice" name="3" style="left: 123px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-123px;"></div><div class="nivo-slice" name="4" style="left: 164px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-164px;"></div><div class="nivo-slice" name="5" style="left: 205px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-205px;"></div><div class="nivo-slice" name="6" style="left: 246px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-246px;"></div><div class="nivo-slice" name="7" style="left: 287px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-287px;"></div><div class="nivo-slice" name="8" style="left: 328px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-328px;"></div><div class="nivo-slice" name="9" style="left: 369px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-369px;"></div><div class="nivo-slice" name="10" style="left: 410px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-410px;"></div><div class="nivo-slice" name="11" style="left: 451px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-451px;"></div><div class="nivo-slice" name="12" style="left: 492px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-492px;"></div><div class="nivo-slice" name="13" style="left: 533px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-533px;"></div><div class="nivo-slice" name="14" style="left: 574px; width: 40.713px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-574px;"></div></div><div class="nivo-controlNav"><a class="nivo-control active" rel="”nofollow”" rels="0" aria-label="Show slide 1 of 2">1</a><a class="nivo-control" rel="”nofollow”" rels="1" aria-label="Show slide 2 of 2">2</a></div></div>

    </div>
</div>Tarpon Point located in Cape Coral, Florida is a stunning, mixed use property that comprises of a resort hotel, restaurants, retail and office buildings, a marina, condominiums and waterfront land. This was a bank foreclosure property to be liquidated at a reduced price of $ 60 M in 2010.<p></p>
<p>Our goal was to acquire the asset and manage it, providing services, interim hotel management, condominium conversion, the sale of marina and slips, sale of condominiums and undeveloped entitled land. &nbsp;During this tough recessionary period there was a limited time frame to raise the equity and we assigned it to one of our financial partners who paid cash with a six week close of escrow.</p>
<p>Tarpon Point consisted of 4 main components:</p>
<p><strong>The Resort Hotel At Marina Village</strong><br>
Hotel and services – totaling 320,000 SF</p>
<p><strong>Marina</strong><br>
Docking and Fueling -175 boat slips</p>
<p><strong>Tarpon Landings</strong><br>
Luxury Condominiums – 205,000 SF</p>
<p><strong>Undeveloped Land</strong><br>
Single family and multi-family development – 18.7 acres</p>
<p><strong>Asking Price $ 60 M</strong><br>
<strong>Disposition Price $ 44 M</strong></p>
<p><a id="02" tabindex="-1"></a></p>
<hr>
<p>&nbsp;</p>
<h2>Office Building Portfolio, Los Angeles, CA</h2>
<p><strong>Class A office portfolio located in Woodland Hills, Van Nuys and Agoura Hills</strong></p>
<p></p><div id="metaslider-id-79" style="width: 100%;" class="ml-slider-3-70-2 metaslider metaslider-nivo metaslider-79 ml-slider ms-theme-default" role="region" aria-roledescription="Slideshow" aria-label="Office Bldg" tabindex="1">
    <div id="metaslider_container_79">
        <div class="slider-wrapper theme-default"><div class="ribbon"></div><div id="metaslider_79" class="nivoSlider"><img decoding="async" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" height="300" width="640" alt="" class="slider-79 slide-81" style="width: 614.713px; visibility: hidden; display: inline;"><img loading="lazy" decoding="async" src="https://mulhollandequity.com/wp-content/uploads/2016/02/TheChatea-NightShot-640x300.jpg" height="300" width="640" alt="" class="slider-79 slide-80" style="width: 614.713px; visibility: hidden; display: inline;"><img class="nivo-main-image" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" alt="" style="height: 288.138px; width: 614.713px;"><div class="nivo-caption"></div><div class="nivo-directionNav"><a class="nivo-prevNav" rel="nofollow" aria-label="Previous Slide">&lt;</a><a class="nivo-nextNav" rel="nofollow" aria-label="Next Slide">&gt;</a></div><div class="nivo-slice" name="0" style="left: 0px; width: 614.713px; height: 288.138px; opacity: 1; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-0px;"></div><div class="nivo-slice" name="1" style="left: 41px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-41px;"></div><div class="nivo-slice" name="2" style="left: 82px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-82px;"></div><div class="nivo-slice" name="3" style="left: 123px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-123px;"></div><div class="nivo-slice" name="4" style="left: 164px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-164px;"></div><div class="nivo-slice" name="5" style="left: 205px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-205px;"></div><div class="nivo-slice" name="6" style="left: 246px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-246px;"></div><div class="nivo-slice" name="7" style="left: 287px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-287px;"></div><div class="nivo-slice" name="8" style="left: 328px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-328px;"></div><div class="nivo-slice" name="9" style="left: 369px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-369px;"></div><div class="nivo-slice" name="10" style="left: 410px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-410px;"></div><div class="nivo-slice" name="11" style="left: 451px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-451px;"></div><div class="nivo-slice" name="12" style="left: 492px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-492px;"></div><div class="nivo-slice" name="13" style="left: 533px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-533px;"></div><div class="nivo-slice" name="14" style="left: 574px; width: 40.713px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-574px;"></div></div><div class="nivo-controlNav"><a class="nivo-control active" rel="”nofollow”" rels="0" aria-label="Show slide 1 of 2">1</a><a class="nivo-control" rel="”nofollow”" rels="1" aria-label="Show slide 2 of 2">2</a></div></div>

    </div>
</div>Three great class A buildings with exceptional locations serving the San Fernando Valley business community. Professional office buildings, provides tenants with central access to prime amenities including shopping, restaurants and other services. Positioned as full service building with subterranean parking and easy access to 101 or 405 freeways.<p></p>
<p>Acquired over a period of 5 years, the Chateau, Woodland Hills featured here was sold in 2011 to investment company.</p>
<p>MEP assisted in raising the equity and provides management services for the remaining two real estate assets.</p>
<p><strong>Acquisition Price $ 28 M</strong><br>
<strong>Disposition of Chateau Bldg $ 12 M</strong><br>
<strong>Current Value of Assets $ 27.5 M</strong></p>
<p><a id="03" tabindex="-1"></a></p>
<hr>
<p>&nbsp;</p>
<h2>Luxury Residence, Los Angeles, CA</h2>
<p></p><div id="metaslider-id-83" style="width: 100%;" class="ml-slider-3-70-2 metaslider metaslider-nivo metaslider-83 ml-slider ms-theme-default" role="region" aria-roledescription="Slideshow" aria-label="Luxury Residence" tabindex="1">
    <div id="metaslider_container_83">
        <div class="slider-wrapper theme-default"><div class="ribbon"></div><div id="metaslider_83" class="nivoSlider"><img loading="lazy" decoding="async" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" height="300" width="640" alt="" class="slider-83 slide-85" style="width: 614.713px; visibility: hidden; display: inline;"><img loading="lazy" decoding="async" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-nightview-640x300.jpg" height="300" width="640" alt="" class="slider-83 slide-84" style="width: 614.713px; visibility: hidden; display: inline;"><img class="nivo-main-image" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" alt="" style="height: 288.138px; width: 614.713px;"><div class="nivo-caption"></div><div class="nivo-directionNav"><a class="nivo-prevNav" rel="nofollow" aria-label="Previous Slide">&lt;</a><a class="nivo-nextNav" rel="nofollow" aria-label="Next Slide">&gt;</a></div><div class="nivo-slice" name="0" style="left: 0px; width: 614.713px; height: 288.138px; opacity: 1; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-0px;"></div><div class="nivo-slice" name="1" style="left: 41px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-41px;"></div><div class="nivo-slice" name="2" style="left: 82px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-82px;"></div><div class="nivo-slice" name="3" style="left: 123px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-123px;"></div><div class="nivo-slice" name="4" style="left: 164px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-164px;"></div><div class="nivo-slice" name="5" style="left: 205px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-205px;"></div><div class="nivo-slice" name="6" style="left: 246px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-246px;"></div><div class="nivo-slice" name="7" style="left: 287px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-287px;"></div><div class="nivo-slice" name="8" style="left: 328px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-328px;"></div><div class="nivo-slice" name="9" style="left: 369px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-369px;"></div><div class="nivo-slice" name="10" style="left: 410px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-410px;"></div><div class="nivo-slice" name="11" style="left: 451px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-451px;"></div><div class="nivo-slice" name="12" style="left: 492px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-492px;"></div><div class="nivo-slice" name="13" style="left: 533px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-533px;"></div><div class="nivo-slice" name="14" style="left: 574px; width: 40.713px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-574px;"></div></div><div class="nivo-controlNav"><a class="nivo-control active" rel="”nofollow”" rels="0" aria-label="Show slide 1 of 2">1</a><a class="nivo-control" rel="”nofollow”" rels="1" aria-label="Show slide 2 of 2">2</a></div></div>

    </div>
</div>Located in the San Fernando Valley, MEP has forged forward into the construction of single family homes. Recently completed in Sherman Oaks, a exquisite example of ultra modern architecture, custom Fleetwood Windows and a 28 foot waterfall cascades into a blue lagoon.<p></p>
<p><strong>Listed Price $2,995,000</strong></p>
<p>Recent developed homes include Woodland Hills and Calabasas</p>




                         <!--Start Comment box-->


<!-- You can start editing here. -->

<div id="commentsbox">







</div>

                        <!--End Comment box-->
                        </div>
'''

# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, 'html.parser')

# Extract the property details
properties = []

# Loop through each h2 tag to get property names and their details
for h2_tag in soup.find_all('h2'):
    property_name = h2_tag.get_text()
    details = []

    # Get the next sibling elements until another h2 or hr tag is found
    for sibling in h2_tag.find_next_siblings():
        if sibling.name in ['h2', 'hr']:
            break
        details.append(sibling.get_text(strip=True))

    property_details = ' '.join(details)
    properties.append({'Property Name': property_name, 'Details': property_details})

# Create a DataFrame from the list of dictionaries
df = pd.DataFrame(properties)

# Save the DataFrame to a CSV file
csv_file_path = '/C:/Users/chinmay/Desktop/chinmay/MMCRE_assignment1/property_details.csv'
df.to_csv(csv_file_path, index=False)

# Download the CSV file
files.download(csv_file_path)

print("Property details have been saved to 'property_details.csv' and is ready for download.")



OSError: Cannot save file into a non-existent directory: '/C:/Users/chinmay/Desktop/chinmay/MMCRE_assignment1'

In [3]:
from bs4 import BeautifulSoup
import pandas as pd
from google.colab import files

# Example HTML content
html_content = '''
<div class="content-bar">
                                                    <p>Over the past two decades, we have transacted and disposed of multiple real estate asset transactions multi family, office, strip centers, industrial, hospitality, and single family homes. Moving forward, the company will evolve and engage in the development of real estate assets. We have established “funds” for these real estate assets, and these funds have performed well, delivering annual high double-digit returns.</p>
<p>Here is a partial list of the portfolios and projects we have facilitated and/or for which we have provided advisory services:</p>
<p>The Ventura Equity Group LLC – A SFR fund for flipping houses in LA County – 2011-2015<br>
The Wilshire Capital Group LLC – A SFR fund for flipping houses in LA County – 2010-2014<br>
Disposition 200 SFR portfolio, 2014<br>
Disposition 115 SFR portfolio, 2013<br>
<a id="01" tabindex="-1"></a></p>
<h2>Tarpon Point, Cape Coral, FL</h2>
<p></p><div id="metaslider-id-72" style="width: 100%;" class="ml-slider-3-70-2 metaslider metaslider-nivo metaslider-72 ml-slider ms-theme-default" role="region" aria-roledescription="Slideshow" aria-label="Tarpon Point" tabindex="1">
    <div id="metaslider_container_72">
        <div class="slider-wrapper theme-default"><div class="ribbon"></div><div id="metaslider_72" class="nivoSlider"><img fetchpriority="high" decoding="async" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" height="300" width="640" alt="" class="slider-72 slide-73" style="width: 614.713px; visibility: hidden; display: inline;"><img decoding="async" src="https://mulhollandequity.com/wp-content/uploads/2016/02/TarponPoint-resized-640x300.jpg" height="300" width="640" alt="" class="slider-72 slide-76" style="width: 614.713px; visibility: hidden; display: inline;"><img class="nivo-main-image" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" alt="" style="height: 288.138px; width: 614.713px;"><div class="nivo-caption"></div><div class="nivo-directionNav"><a class="nivo-prevNav" rel="nofollow" aria-label="Previous Slide">&lt;</a><a class="nivo-nextNav" rel="nofollow" aria-label="Next Slide">&gt;</a></div><div class="nivo-slice" name="0" style="left: 0px; width: 614.713px; height: 288.138px; opacity: 1; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-0px;"></div><div class="nivo-slice" name="1" style="left: 41px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-41px;"></div><div class="nivo-slice" name="2" style="left: 82px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-82px;"></div><div class="nivo-slice" name="3" style="left: 123px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-123px;"></div><div class="nivo-slice" name="4" style="left: 164px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-164px;"></div><div class="nivo-slice" name="5" style="left: 205px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-205px;"></div><div class="nivo-slice" name="6" style="left: 246px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-246px;"></div><div class="nivo-slice" name="7" style="left: 287px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-287px;"></div><div class="nivo-slice" name="8" style="left: 328px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-328px;"></div><div class="nivo-slice" name="9" style="left: 369px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-369px;"></div><div class="nivo-slice" name="10" style="left: 410px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-410px;"></div><div class="nivo-slice" name="11" style="left: 451px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-451px;"></div><div class="nivo-slice" name="12" style="left: 492px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-492px;"></div><div class="nivo-slice" name="13" style="left: 533px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-533px;"></div><div class="nivo-slice" name="14" style="left: 574px; width: 40.713px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/tarp1-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-574px;"></div></div><div class="nivo-controlNav"><a class="nivo-control active" rel="”nofollow”" rels="0" aria-label="Show slide 1 of 2">1</a><a class="nivo-control" rel="”nofollow”" rels="1" aria-label="Show slide 2 of 2">2</a></div></div>

    </div>
</div>Tarpon Point located in Cape Coral, Florida is a stunning, mixed use property that comprises of a resort hotel, restaurants, retail and office buildings, a marina, condominiums and waterfront land. This was a bank foreclosure property to be liquidated at a reduced price of $ 60 M in 2010.<p></p>
<p>Our goal was to acquire the asset and manage it, providing services, interim hotel management, condominium conversion, the sale of marina and slips, sale of condominiums and undeveloped entitled land. &nbsp;During this tough recessionary period there was a limited time frame to raise the equity and we assigned it to one of our financial partners who paid cash with a six week close of escrow.</p>
<p>Tarpon Point consisted of 4 main components:</p>
<p><strong>The Resort Hotel At Marina Village</strong><br>
Hotel and services – totaling 320,000 SF</p>
<p><strong>Marina</strong><br>
Docking and Fueling -175 boat slips</p>
<p><strong>Tarpon Landings</strong><br>
Luxury Condominiums – 205,000 SF</p>
<p><strong>Undeveloped Land</strong><br>
Single family and multi-family development – 18.7 acres</p>
<p><strong>Asking Price $ 60 M</strong><br>
<strong>Disposition Price $ 44 M</strong></p>
<p><a id="02" tabindex="-1"></a></p>
<hr>
<p>&nbsp;</p>
<h2>Office Building Portfolio, Los Angeles, CA</h2>
<p><strong>Class A office portfolio located in Woodland Hills, Van Nuys and Agoura Hills</strong></p>
<p></p><div id="metaslider-id-79" style="width: 100%;" class="ml-slider-3-70-2 metaslider metaslider-nivo metaslider-79 ml-slider ms-theme-default" role="region" aria-roledescription="Slideshow" aria-label="Office Bldg" tabindex="1">
    <div id="metaslider_container_79">
        <div class="slider-wrapper theme-default"><div class="ribbon"></div><div id="metaslider_79" class="nivoSlider"><img decoding="async" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" height="300" width="640" alt="" class="slider-79 slide-81" style="width: 614.713px; visibility: hidden; display: inline;"><img loading="lazy" decoding="async" src="https://mulhollandequity.com/wp-content/uploads/2016/02/TheChatea-NightShot-640x300.jpg" height="300" width="640" alt="" class="slider-79 slide-80" style="width: 614.713px; visibility: hidden; display: inline;"><img class="nivo-main-image" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" alt="" style="height: 288.138px; width: 614.713px;"><div class="nivo-caption"></div><div class="nivo-directionNav"><a class="nivo-prevNav" rel="nofollow" aria-label="Previous Slide">&lt;</a><a class="nivo-nextNav" rel="nofollow" aria-label="Next Slide">&gt;</a></div><div class="nivo-slice" name="0" style="left: 0px; width: 614.713px; height: 288.138px; opacity: 1; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-0px;"></div><div class="nivo-slice" name="1" style="left: 41px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-41px;"></div><div class="nivo-slice" name="2" style="left: 82px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-82px;"></div><div class="nivo-slice" name="3" style="left: 123px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-123px;"></div><div class="nivo-slice" name="4" style="left: 164px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-164px;"></div><div class="nivo-slice" name="5" style="left: 205px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-205px;"></div><div class="nivo-slice" name="6" style="left: 246px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-246px;"></div><div class="nivo-slice" name="7" style="left: 287px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-287px;"></div><div class="nivo-slice" name="8" style="left: 328px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-328px;"></div><div class="nivo-slice" name="9" style="left: 369px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-369px;"></div><div class="nivo-slice" name="10" style="left: 410px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-410px;"></div><div class="nivo-slice" name="11" style="left: 451px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-451px;"></div><div class="nivo-slice" name="12" style="left: 492px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-492px;"></div><div class="nivo-slice" name="13" style="left: 533px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-533px;"></div><div class="nivo-slice" name="14" style="left: 574px; width: 40.713px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Chateau-day-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-574px;"></div></div><div class="nivo-controlNav"><a class="nivo-control active" rel="”nofollow”" rels="0" aria-label="Show slide 1 of 2">1</a><a class="nivo-control" rel="”nofollow”" rels="1" aria-label="Show slide 2 of 2">2</a></div></div>

    </div>
</div>Three great class A buildings with exceptional locations serving the San Fernando Valley business community. Professional office buildings, provides tenants with central access to prime amenities including shopping, restaurants and other services. Positioned as full service building with subterranean parking and easy access to 101 or 405 freeways.<p></p>
<p>Acquired over a period of 5 years, the Chateau, Woodland Hills featured here was sold in 2011 to investment company.</p>
<p>MEP assisted in raising the equity and provides management services for the remaining two real estate assets.</p>
<p><strong>Acquisition Price $ 28 M</strong><br>
<strong>Disposition of Chateau Bldg $ 12 M</strong><br>
<strong>Current Value of Assets $ 27.5 M</strong></p>
<p><a id="03" tabindex="-1"></a></p>
<hr>
<p>&nbsp;</p>
<h2>Luxury Residence, Los Angeles, CA</h2>
<p></p><div id="metaslider-id-83" style="width: 100%;" class="ml-slider-3-70-2 metaslider metaslider-nivo metaslider-83 ml-slider ms-theme-default" role="region" aria-roledescription="Slideshow" aria-label="Luxury Residence" tabindex="1">
    <div id="metaslider_container_83">
        <div class="slider-wrapper theme-default"><div class="ribbon"></div><div id="metaslider_83" class="nivoSlider"><img loading="lazy" decoding="async" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" height="300" width="640" alt="" class="slider-83 slide-85" style="width: 614.713px; visibility: hidden; display: inline;"><img loading="lazy" decoding="async" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-nightview-640x300.jpg" height="300" width="640" alt="" class="slider-83 slide-84" style="width: 614.713px; visibility: hidden; display: inline;"><img class="nivo-main-image" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" alt="" style="height: 288.138px; width: 614.713px;"><div class="nivo-caption"></div><div class="nivo-directionNav"><a class="nivo-prevNav" rel="nofollow" aria-label="Previous Slide">&lt;</a><a class="nivo-nextNav" rel="nofollow" aria-label="Next Slide">&gt;</a></div><div class="nivo-slice" name="0" style="left: 0px; width: 614.713px; height: 288.138px; opacity: 1; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-0px;"></div><div class="nivo-slice" name="1" style="left: 41px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-41px;"></div><div class="nivo-slice" name="2" style="left: 82px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-82px;"></div><div class="nivo-slice" name="3" style="left: 123px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-123px;"></div><div class="nivo-slice" name="4" style="left: 164px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-164px;"></div><div class="nivo-slice" name="5" style="left: 205px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-205px;"></div><div class="nivo-slice" name="6" style="left: 246px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-246px;"></div><div class="nivo-slice" name="7" style="left: 287px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-287px;"></div><div class="nivo-slice" name="8" style="left: 328px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-328px;"></div><div class="nivo-slice" name="9" style="left: 369px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-369px;"></div><div class="nivo-slice" name="10" style="left: 410px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-410px;"></div><div class="nivo-slice" name="11" style="left: 451px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-451px;"></div><div class="nivo-slice" name="12" style="left: 492px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-492px;"></div><div class="nivo-slice" name="13" style="left: 533px; width: 41px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-533px;"></div><div class="nivo-slice" name="14" style="left: 574px; width: 40.713px; height: 288.138px; opacity: 0; overflow: hidden;"><img alt="" src="https://mulhollandequity.com/wp-content/uploads/2016/02/Stansbury-front-640x300.jpg" style="position:absolute; width:614.713px; height:auto; display:block !important; top:0; left:-574px;"></div></div><div class="nivo-controlNav"><a class="nivo-control active" rel="”nofollow”" rels="0" aria-label="Show slide 1 of 2">1</a><a class="nivo-control" rel="”nofollow”" rels="1" aria-label="Show slide 2 of 2">2</a></div></div>

    </div>
</div>Located in the San Fernando Valley, MEP has forged forward into the construction of single family homes. Recently completed in Sherman Oaks, a exquisite example of ultra modern architecture, custom Fleetwood Windows and a 28 foot waterfall cascades into a blue lagoon.<p></p>
<p><strong>Listed Price $2,995,000</strong></p>
<p>Recent developed homes include Woodland Hills and Calabasas</p>




                         <!--Start Comment box-->


<!-- You can start editing here. -->

<div id="commentsbox">







</div>

                        <!--End Comment box-->
                        </div>
'''

# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, 'html.parser')

# Extract the property details
properties = []

# Loop through each h2 tag to get property names and their details
for h2_tag in soup.find_all('h2'):
    property_name = h2_tag.get_text()
    details = []

    # Get the next sibling elements until another h2 or hr tag is found
    for sibling in h2_tag.find_next_siblings():
        if sibling.name in ['h2', 'hr']:
            break
        details.append(sibling.get_text(strip=True))

    property_details = ' '.join(details)
    properties.append({'Property Name': property_name, 'Details': property_details})

# Create a DataFrame from the list of dictionaries
df = pd.DataFrame(properties)

# Save the DataFrame to a CSV file
csv_file_path = '/content/property_details.csv'

# Save to CSV
df.to_csv(csv_file_path, index=False)

# Trigger download
files.download(csv_file_path)

print("Property details have been saved to 'property_details.csv' and the download has been triggered.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Property details have been saved to 'property_details.csv' and the download has been triggered.


In [5]:
!pip install PyMuPDF
import fitz  # PyMuPDF
import pandas as pd
from google.colab import files

# Function to extract text from a PDF
def extract_text_from_pdf(pdf_file):
    doc = fitz.open(pdf_file)
    text = ""
    for page_num in range(doc.page_count):
        page = doc.load_page(page_num)
        text += page.get_text()
    return text

# Function to parse the extracted text into a structured format
def parse_pdf_text(text):
    data = []
    current_state = None

    lines = text.splitlines()
    for line in lines:
        if line.isupper() and len(line.split()) == 1:  # Assuming state names are in uppercase
            current_state = line
        elif 'Center' in line and 'Location' in line and 'Anchor Tenants' in line and 'GLA' in line:
            continue  # Skip the header line
        elif current_state and line.strip():  # Parse property details
            details = line.split(',')
            if len(details) == 4:
                data.append([current_state] + details)

    return data

# Function to convert parsed data into a DataFrame and save as CSV
def save_to_csv(data, filename):
    df = pd.DataFrame(data, columns=['State', 'Center', 'Location', 'Anchor Tenants', 'GLA'])
    df.to_csv(filename, index=False)
    files.download(filename)  # Trigger download

# Main workflow
# Upload PDF file
uploaded = files.upload()

# Extract the first file name from the uploaded files
pdf_file = list(uploaded.keys())[0]

output_csv = 'properties_portfolio.csv'

# Extract text from PDF
pdf_text = extract_text_from_pdf(pdf_file)

# Parse the text
parsed_data = parse_pdf_text(pdf_text)

# Save to CSV and trigger download
save_to_csv(parsed_data, output_csv)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 34.6 MB/s eta 0:00:00


Saving KaminRealty_Portfolio_2024.pdf to KaminRealty_Portfolio_2024.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
import fitz  # PyMuPDF
import pandas as pd
from google.colab import files

# Function to extract text from a PDF
def extract_text_from_pdf(pdf_file):
    doc = fitz.open(pdf_file)
    text = ""
    for page_num in range(doc.page_count):
        page = doc.load_page(page_num)
        text += page.get_text()
    return text

# Function to parse the extracted text into a structured format
def parse_pdf_text(text):
    data = []
    current_state = None

    lines = text.splitlines()
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.isupper() and len(line.split()) == 1:  # State name (assumed to be in uppercase)
            current_state = line
        elif current_state and 'Center Location Anchor Tenants GLA' not in line and line:
            center = line
            location = lines[i + 1].strip() if (i + 1) < len(lines) else ""
            anchor_tenants = lines[i + 2].strip() if (i + 2) < len(lines) else ""
            gla = lines[i + 3].strip() if (i + 3) < len(lines) else ""

            data.append([current_state, center, location, anchor_tenants, gla])

            i += 3  # Move to the next property block
        i += 1

    return data

# Function to convert parsed data into a DataFrame and save as CSV
def save_to_csv(data, filename):
    df = pd.DataFrame(data, columns=['State', 'Center', 'Location', 'Anchor Tenants', 'GLA'])
    df.to_csv(filename, index=False)
    files.download(filename)  # Trigger download

# Main workflow
# Upload PDF file
uploaded = files.upload()

# Extract the first file name from the uploaded files
pdf_file = list(uploaded.keys())[0]

output_csv = 'properties_portfolio.csv'

# Extract text from PDF
pdf_text = extract_text_from_pdf(pdf_file)

# Parse the text
parsed_data = parse_pdf_text(pdf_text)

# Save to CSV and trigger download
save_to_csv(parsed_data, output_csv)



Saving KaminRealty_Portfolio_2024.pdf to KaminRealty_Portfolio_2024 (2).pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import fitz  # PyMuPDF
import pandas as pd
from google.colab import files

# Function to extract text from a PDF
def extract_text_from_pdf(pdf_file):
    doc = fitz.open(pdf_file)
    text = ""
    for page_num in range(doc.page_count):
        page = doc.load_page(page_num)
        text += page.get_text()
    return text

# Function to parse the extracted text into a structured format
def parse_pdf_text(text):
    data = []
    current_state = None
    state_names = [
         "ALABAMA", "ALASKA", "ARIZONA", "ARKANSAS", "CALIFORNIA",
    "COLORADO", "CONNECTICUT", "DELAWARE", "FLORIDA", "GEORGIA",
    "HAWAII", "IDAHO", "ILLINOIS", "INDIANA", "IOWA",
    "KANSAS", "KENTUCKY", "LOUISIANA", "MAINE", "MARYLAND",
    "MASSACHUSETTS", "MICHIGAN", "MINNESOTA", "MISSISSIPPI", "MISSOURI",
    "MONTANA", "NEBRASKA", "NEVADA", "NEW HAMPSHIRE", "NEW JERSEY",
    "NEW MEXICO", "NEW YORK", "NORTH CAROLINA", "NORTH DAKOTA", "OHIO",
    "OKLAHOMA", "OREGON", "PENNSYLVANIA", "RHODE ISLAND", "SOUTH CAROLINA",
    "SOUTH DAKOTA", "TENNESSEE", "TEXAS", "UTAH", "VERMONT",
    "VIRGINIA", "WASHINGTON", "WEST VIRGINIA", "WISCONSIN", "WYOMING"
        # Add more states as needed
    ]

    lines = text.splitlines()
    i = 0
    while i < len(lines):
        line = lines[i].strip()

        if line in state_names:  # Match exact state names
            current_state = line
        elif current_state and line and line != 'Center Location Anchor Tenants GLA':
            center = line
            location = lines[i + 1].strip() if (i + 1) < len(lines) else ""
            anchor_tenants = lines[i + 2].strip() if (i + 2) < len(lines) else ""
            gla = lines[i + 3].strip() if (i + 3) < len(lines) else ""

            data.append([current_state, center, location, anchor_tenants, gla])

            i += 3  # Move to the next property block
        i += 1

    return data

# Function to convert parsed data into a DataFrame and save as CSV
def save_to_csv(data, filename):
    df = pd.DataFrame(data, columns=['State', 'Center', 'Location', 'Anchor Tenants', 'GLA'])
    df.to_csv(filename, index=False)
    files.download(filename)  # Trigger download

# Main workflow
# Upload PDF file
uploaded = files.upload()

# Extract the first file name from the uploaded files
pdf_file = list(uploaded.keys())[0]

output_csv = 'properties_portfolio.csv'

# Extract text from PDF
pdf_text = extract_text_from_pdf(pdf_file)

# Parse the text
parsed_data = parse_pdf_text(pdf_text)

# Save to CSV and trigger download
save_to_csv(parsed_data, output_csv)


Saving KaminRealty_Portfolio_2024.pdf to KaminRealty_Portfolio_2024 (3).pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
!pip install PyPDF2
import re
import csv
import PyPDF2
from google.colab import files

# Function to parse the PDF and extract data
def extract_property_details_from_pdf(pdf_path):
    # List of state names to look for
    state_names = [
        "ALABAMA", "ALASKA", "ARIZONA", "ARKANSAS", "CALIFORNIA",
        "COLORADO", "CONNECTICUT", "DELAWARE", "FLORIDA", "GEORGIA",
        "HAWAII", "IDAHO", "ILLINOIS", "INDIANA", "IOWA",
        "KANSAS", "KENTUCKY", "LOUISIANA", "MAINE", "MARYLAND",
        "MASSACHUSETTS", "MICHIGAN", "MINNESOTA", "MISSISSIPPI", "MISSOURI",
        "MONTANA", "NEBRASKA", "NEVADA", "NEW HAMPSHIRE", "NEW JERSEY",
        "NEW MEXICO", "NEW YORK", "NORTH CAROLINA", "NORTH DAKOTA", "OHIO",
        "OKLAHOMA", "OREGON", "PENNSYLVANIA", "RHODE ISLAND", "SOUTH CAROLINA",
        "SOUTH DAKOTA", "TENNESSEE", "TEXAS", "UTAH", "VERMONT",
        "VIRGINIA", "WASHINGTON", "WEST VIRGINIA", "WISCONSIN", "WYOMING"
    ]

    # Read PDF
    pdf_reader = PyPDF2.PdfReader(pdf_path)
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text()

    # Split the text by lines
    lines = text.splitlines()

    properties = []
    current_state = None
    current_center = None
    current_location = None

    for line in lines:
        line = line.strip()

        # Check if the line is a state name
        if line in state_names:
            current_state = line
            continue

        # If the line is a center (first column), extract it
        if current_state and re.match(r"^[\w\s\(\)\-']+$", line):
            if not re.match(r"^\d+|[A-Z]{2}\s\d+$", line):  # Avoid matching location lines or zip codes
                current_center = line
                continue

        # Check if the line contains location and other details
        match = re.match(r"^(.*?)(\d{4,}\s.*)$", line)
        if match and current_center:
            location = match.group(1).strip()
            rest_of_details = match.group(2).strip()

            # Split rest of details into Anchor Tenants and GLA
            rest_match = re.match(r"^(.*?)\s(\d{2,3},?\d{0,3})$", rest_of_details)
            if rest_match:
                anchor_tenants = rest_match.group(1).strip()
                gla = rest_match.group(2).strip()
                properties.append([current_state, current_center, location, anchor_tenants, gla])
                continue

        # If the line appears to be a continuation of the location, update current_location
        if current_center and not re.match(r"^\d+\s", line) and re.match(r".*,\s*[A-Z]{2}", line):
            current_location = line
            continue

    return properties

# Function to save extracted properties to CSV
def save_to_csv(properties, output_csv_path):
    with open(output_csv_path, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["State", "Center", "Location", "Anchor Tenants", "GLA"])
        writer.writerows(properties)

    # Trigger file download in Google Colab
    files.download(output_csv_path)

# Example usage
uploaded = files.upload()

for file_name in uploaded.keys():
    extracted_properties = extract_property_details_from_pdf(file_name)
    save_to_csv(extracted_properties, "extracted_properties.csv")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.0 MB/s eta 0:00:00


Saving KaminRealty_Portfolio_2024.pdf to KaminRealty_Portfolio_2024 (4).pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>